# Task 2 — Data Integration and Cleaning

**COMP5339 Data Engineering · Assignment 1 · EV Charger Data Integration & Augmentation**

---

This notebook cleans the Transport for NSW charger list retrieved in Task 1 and joins it to the
ABS SA4 boundaries, so that every charger carries the SA4 region it lies in. It addresses the data
quality issues that Task 1's profile identified — missing values, duplicate records, inconsistent
operator names and inconsistent charger attributes — and records every change in a cleaning ledger.

### How to run

Run `01_data_acquisition.ipynb` first: this notebook reads the files it downloaded, locating them
through `data/raw/manifest.json`, and stops with a message if they are missing. Then run this
notebook from top to bottom (**Kernel → Restart Kernel and Run All Cells**). A run takes a few
seconds and needs no network access, except to download the DuckDB spatial extension if Task 1 has
not already installed it.

### Contents

1. [Approach](#t2-approach)
2. [Setup](#t2-setup)
3. [Structural cleaning](#t2-structure)
4. [Operator names](#t2-operators)
5. [Charger type and status](#t2-type)
6. [Charger power ratings](#t2-rating)
7. [Addresses and postcodes](#t2-address)
8. [Coordinates and sites](#t2-coords)
9. [Duplicates and reconciliation](#t2-dupes), including the same charger in two feeds
10. [Derived rating fields](#t2-ratingfields)
11. [Cross-field consistency](#t2-consistency)
12. [Spatial integration with ASGS SA4](#t2-spatial)
13. [Validation and summary](#t2-validate)
14. [Outputs](#t2-outputs)

### Scope

This notebook covers **Task 2 only**. Its outputs in `data/interim/` are the inputs to
`03_data_augmentation.ipynb` (Task 3) and `04_data_storage.ipynb` (Task 4).

<a id="t2-approach"></a>
## 1. Approach

Task 1 established *what* is wrong with the source data. This notebook fixes it and
joins the result to the ABS geography, producing the cleaned dataset that Tasks 3 and 4 build on.

Three principles run through everything below.

**Nothing is discarded silently.** Every transformation appends an entry to a *cleaning ledger*
recording what was done and how many records it touched. That ledger is written to
`data/interim/cleaning_log.json` and printed at the end, so every claim about the cleaning can be
checked: "duplicates were removed" is worth much less than "no exact duplicates found; 12
conflicting pairs merged under stated rules".

**Repair where a rule can be justified; flag where it cannot.** A trailing space in `'BP Australia '`
has one obviously correct fix. A locality recorded as `Sydney` for an address in Riverwood does
not — inventing the true suburb would be fabricating data. Problems of the second kind get a
boolean `*_flag` column, so a downstream query can exclude unreliable records instead of trusting
a value someone guessed. The cleaned output therefore carries nine such flags alongside the data.

**Order matters.** Whitespace is normalised before anything compares strings, deduplication runs
after operator names are canonicalised (otherwise `Tesla` and `Tesla Motors ` at the same site look
like two different chargers), and the derived power columns are computed *after* deduplication so
they always describe the record that actually survived. Each of those orderings is noted where it
occurs.

### What Task 2 produces

| File | Contents |
|---|---|
| `data/interim/ev_chargers_clean.csv` | one row per charger, cleaned, with SA4 fields and quality flags |
| `data/interim/charger_power_ratings.csv` | one row per connector group, unpacking the compound ratings |
| `data/interim/probable_duplicates.csv` | pairs of records that are probably one charger listed in two feeds (section 9.1) |
| `data/interim/sa4_regions_nsw.csv` | the 30 NSW SA4 codes (28 regions plus 2 ABS non-spatial codes), the region dimension in Task 4 |
| `data/interim/cleaning_log.json` | the cleaning ledger — every step and the records it affected |

<a id="t2-setup"></a>
## 2. Setup

Task 2 depends on Task 1's *outputs*, never on its in-memory variables. The paths of the downloaded
files are read from `data/raw/manifest.json`, the provenance record Task 1 writes, so this notebook
always cleans exactly the files Task 1 retrieved and checksummed. As in Task 1, the packages are
installed from within the notebook, so it runs on a fresh machine.

In [1]:
%pip install -q "pandas>=2.2" "numpy>=1.26" "duckdb>=1.5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

# --- Locate the Task 1 outputs ---------------------------------------------
# Paths come from the manifest Task 1 wrote, so Task 2 depends on Task 1's
# outputs rather than on its variables still being in memory.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
MANIFEST_PATH = RAW_DIR / "manifest.json"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "data/raw/manifest.json is missing - run 01_data_acquisition.ipynb first."
    )
_artefacts = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))["artefacts"]
EV_CSV_PATH = PROJECT_ROOT / _artefacts["tfnsw_ev_charging_locations"]["path"]
SA4_SHAPEFILE_PATH = PROJECT_ROOT / _artefacts["abs_asgs_sa4_boundaries"]["shapefile"]
for required in (EV_CSV_PATH, SA4_SHAPEFILE_PATH):
    if not required.exists():
        raise FileNotFoundError(
            f"{required.relative_to(PROJECT_ROOT)} is missing - run 01_data_acquisition.ipynb first."
        )
for directory in (INTERIM_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Task 2 outputs ---------------------------------------------------------
CLEAN_CHARGERS_PATH = INTERIM_DIR / "ev_chargers_clean.csv"
RATINGS_PATH = INTERIM_DIR / "charger_power_ratings.csv"
DUPLICATES_PATH = INTERIM_DIR / "probable_duplicates.csv"
SA4_NSW_PATH = INTERIM_DIR / "sa4_regions_nsw.csv"
CLEANING_LOG_PATH = INTERIM_DIR / "cleaning_log.json"

# --- Reference values used by the cleaning rules ----------------------------
NSW_LAT_RANGE = (-37.6, -28.1)
NSW_LON_RANGE = (140.9, 153.7)
COORDINATE_DECIMALS = 6          # ~0.11 m - finer than the source can justify
NSW_POSTCODE_RANGES = ((1000, 1999), (2000, 2599), (2619, 2899), (2921, 2999))

SOURCE_CRS = "EPSG:7844"         # GDA2020, the datum of the ABS boundaries
TARGET_CRS = "EPSG:4326"         # WGS84, the datum of the TfNSW lat/lon columns

# `cleaning_log` accumulates one entry per transformation: what was done and how
# many records it touched. It is written to disk at the end, so every change the
# cleaning made can be checked and quantified.
cleaning_log = []


def record_step(step: str, detail: str, affected: int) -> None:
    """Append one auditable entry to the cleaning ledger and echo it."""
    cleaning_log.append({"step": step, "detail": detail, "records_affected": int(affected)})
    print(f"  {step:<26} {affected:>6,}  {detail}")


print(f"EV CSV    : {EV_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shapefile : {SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)}")

EV CSV    : data/raw/tfnsw/ev_20251216.csv
Shapefile : data/raw/abs/sa4_shapefile/SA4_2026_AUST_GDA2020.shp


<a id="t2-structure"></a>
## 3. Structural cleaning

### 3.1 Reading the file as text

Task 1 (section 6) read the CSV with pandas' default type inference, which was fine for profiling. For
cleaning it is not: inference is the first place data gets altered without anyone deciding that it
should be. Reading every column as text moves each conversion into a later cell where it is
explicit, ordered, and counted.

In [3]:
# `dtype=str` for every column, deliberately. pandas' type inference is the first
# place data gets silently altered: PCODE would become an integer (losing any
# leading zero and turning the 121 missing values into floats) and OBJECTID would
# become a float for the same reason. Reading everything as text means every
# conversion below is explicit, ordered, and reviewable.
raw = pd.read_csv(EV_CSV_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)

print(f"Loaded {len(raw):,} rows x {raw.shape[1]} columns, all as text")
raw.head(3)

Loaded 1,958 rows x 12 columns, all as text


,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,,,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.26224229,150.8901391,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,,,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.81100405,150.8495966,Blacktown City Council,2766,Existing Fast Chargers
2,,,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.5118739,151.669395,Central Darling Shire Council,2350,TfNSW Regional


### 3.2 Whitespace and missing values

This is the first substantive step because everything downstream compares strings. 733 addresses
contain embedded newlines and two operator names carry a trailing space; until those are gone, a
record cannot be matched against its own duplicate, and `'BP Australia '` is a different operator
from `'BP Australia'`.

Empty strings become proper missing values in the same pass. A `''` and a `NaN` mean the same thing
to a reader and completely different things to `isna()`, `groupby()` and a `NOT NULL` constraint,
so the ambiguity is removed once, here, rather than handled repeatedly later.

In [4]:
def squash_whitespace(value):
    """
    Collapse newlines, tabs and runs of spaces into single spaces, then trim.

    This runs before everything else because every later comparison - dedup,
    operator matching, address parsing - compares strings. 733 addresses contain
    embedded newlines and two operator names carry a trailing space, so without
    this step a large share of the dataset fails to match its own duplicate.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.NA
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if text else pd.NA


df = raw.copy()
text_columns = df.columns.tolist()

blank_cells = int((df[text_columns] == "").sum().sum())
for column in text_columns:
    df[column] = df[column].map(squash_whitespace)
rewritten = int((raw[text_columns].astype(str) != df[text_columns].astype(str)).sum().sum())

print("Normalising whitespace and converting empty strings to NA")
record_step("whitespace normalised", "cells whose text was rewritten", rewritten - blank_cells)
record_step("empty -> NA", "empty strings converted to missing values", blank_cells)

Normalising whitespace and converting empty strings to NA
  whitespace normalised         808  cells whose text was rewritten
  empty -> NA                 3,638  empty strings converted to missing values


### 3.3 Column names and data types

Renaming to `snake_case` is not cosmetic. The source mixes three conventions in twelve columns
(`OBJECTID`, `Station_name`, `LGANAME`), and the destination is a SQL schema in Task 4 where a
single convention avoids quoted identifiers in every query.

The type conversions use `errors="coerce"`, which turns anything unparsable into a missing value
rather than raising. Used carelessly that hides problems, so each conversion counts how many
missing values it *created* and logs them — a coercion that silently destroys data is then
impossible to miss. `Int64` (capital I) rather than `int64` is deliberate too: plain NumPy integers
cannot represent a missing value, so `OBJECTID`, which is null in 1,837 rows, would be forced back
to float.

In [5]:
# snake_case throughout. The source mixes three conventions (`OBJECTID`,
# `Station_name`, `LGANAME`), and the destination is a SQL schema in Task 4 where
# one convention avoids quoted identifiers everywhere.
COLUMN_RENAMES = {
    "OBJECTID": "source_object_id",
    "Station_name": "station_name",
    "Station_address": "station_address_raw",
    "Operator": "operator_raw",
    "Number_of_plugs": "number_of_plugs",
    "Charger_Type": "charger_type_raw",
    "Charger_rating": "charger_rating_raw",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "LGANAME": "lga_name",
    "PCODE": "postcode_reported",
    "Source": "source_feed",
}
df = df.rename(columns=COLUMN_RENAMES)

# `errors="coerce"` turns anything unparsable into NA rather than raising - and
# the number of values it silently created is logged, so a bad coercion can never
# slip past unnoticed. Int64 (capital I) is pandas' nullable integer: plain int64
# cannot hold a missing value and would force these columns back to float.
numeric_specs = {
    "source_object_id": "Int64",
    "number_of_plugs": "Int64",
    "latitude": "float64",
    "longitude": "float64",
}
for column, dtype in numeric_specs.items():
    before_missing = df[column].isna().sum()
    converted = pd.to_numeric(df[column], errors="coerce")
    df[column] = converted.astype(dtype) if dtype == "Int64" else converted
    coerced = int(df[column].isna().sum() - before_missing)
    if coerced:
        record_step("type coercion", f"{column}: unparsable values set to NA", coerced)

# Coordinates rounded to 6 dp. The source carries more decimals than a GPS fix
# justifies, and unrounded floats make two records at one physical site compare
# as different locations.
df["latitude"] = df["latitude"].round(COORDINATE_DECIMALS)
df["longitude"] = df["longitude"].round(COORDINATE_DECIMALS)

df.dtypes.to_frame("dtype")

,dtype
source_object_id,Int64
station_name,object
station_address_raw,object
operator_raw,object
number_of_plugs,Int64
charger_type_raw,object
charger_rating_raw,object
latitude,float64
longitude,float64
lga_name,object


<a id="t2-operators"></a>
## 4. Operator names

Task 1's profile (section 6.3) showed 50 distinct operator strings standing for roughly 35 real operators, with three
distinct causes tangled together. Only one of them can be fixed by a rule.

Case and spacing variants (`ChargeHub` / `Charge Hub`) are mechanical. Short forms and full names
(`BP` / `BP Australia`) require knowing that they are the same company. Truncation at 13–14
characters (`Viva Energy A`, `PLUS ES Manag`) is caused by a fixed-width field somewhere upstream
and is *irreversible from the data alone* — the missing characters simply are not there.

So the repairs live in an explicit lookup table rather than in a similarity heuristic. A heuristic
that merges `Evie` with `Evie Networks` on a shared first word also merges `Charge Hub` with
`Charge OS`, which are unrelated companies, and nothing in the data distinguishes those two cases.
A lookup table is reviewable, arguable, and correct; a fuzzy match is none of the three.

Two decisions are recorded as deliberate non-merges: `Charge Hub` / `Charge OS`, and Porsche's two
programmes. Three truncations have no full form anywhere in the data: `University of`, `Energy Austra` and
`Fast Cities A`. They are flagged rather than guessed at. `Viva Energy A` and `PLUS ES Manag`,
by contrast, could be repaired, because their full forms appear elsewhere in the same release. Any operator absent from the table is printed, so a future TfNSW
release with new operators is noticed rather than passed through unexamined.

In [6]:
# Three problems live in `Operator`, and only the first can be fixed by a rule:
# case/punctuation variants, trailing whitespace, and names truncated at 13-14
# characters by a fixed-width field upstream. Truncation is irreversible from the
# data alone, so the repairs are stated in a lookup table a human can review -
# not inferred by a string-similarity heuristic, which would also merge the
# unrelated 'Charge Hub' and 'Charge OS'.
OPERATOR_CANONICAL = {
    # case and spacing variants
    "chargehub": "ChargeHub",
    "charge hub": "ChargeHub",
    "non-networked": "Non-networked",
    "non networked": "Non-networked",
    # short form and full name of one company
    "bp": "BP Australia",
    "bp australia": "BP Australia",
    "tesla": "Tesla",
    "tesla motors": "Tesla",
    "nrma": "NRMA",
    "nrma electric": "NRMA",
    "evie": "Evie Networks",
    "evie networks": "Evie Networks",
    # cut off mid-word by the upstream fixed-width field
    "plus es": "PLUS ES",
    "plus es manag": "PLUS ES",
    "viva energy a": "Viva Energy Australia",
    "viva energy australia": "Viva Energy Australia",
}

# Deliberately NOT merged:
#   'Charge Hub' vs 'Charge OS'                          unrelated companies
#   'Porsche Destination Charging' vs
#   'Porsche Smart Mobility'                             two distinct programmes
# Truncations that cannot be resolved without an external source are flagged
# rather than guessed at. 'Viva Energy A' and 'PLUS ES Manag' are repaired
# above because their full forms also appear in this release; these three
# have no full form anywhere in it:
UNRESOLVED_TRUNCATIONS = {"university of", "energy austra", "fast cities a"}


def canonical_operator(value):
    """Map a raw operator string to its canonical form via the lookup table."""
    if pd.isna(value):
        return pd.NA
    key = re.sub(r"\s+", " ", str(value)).strip().lower()
    return OPERATOR_CANONICAL.get(key, str(value).strip())


operator_key = df["operator_raw"].fillna("").str.strip().str.lower()
df["operator"] = df["operator_raw"].map(canonical_operator)
df["operator_truncated_flag"] = operator_key.isin(UNRESOLVED_TRUNCATIONS)

print(f"Operator values: {df['operator_raw'].nunique()} raw -> {df['operator'].nunique()} canonical")
record_step("operator canonicalised", "values rewritten to a canonical name",
            int((df["operator"] != df["operator_raw"]).sum()))
record_step("operator truncated", "unresolved truncations flagged for review",
            int(df["operator_truncated_flag"].sum()))

# Anything absent from the lookup table is listed, so the table can be extended
# deliberately when TfNSW publishes a release containing new operators.
unmapped = sorted(set(df.loc[~operator_key.isin(OPERATOR_CANONICAL), "operator"].dropna()))
print(f"\n{len(unmapped)} operator(s) passed through unmapped (whitespace-normalised only):")
print("   ", ", ".join(unmapped))

Operator values: 50 raw -> 42 canonical
  operator canonicalised        181  values rewritten to a canonical name
  operator truncated             19  unresolved truncations flagged for review

34 operator(s) passed through unmapped (whitespace-normalised only):
    360 EV Charge, AXCharge, Alchemy Charge, Ampol, BMW, CasaCharge, Charge OS, ChargePoint, ChargePost, Chargefox, Chargestar, Counties Energy, EV Meter, EVE Australia, EVNet, EVSE, EVUp, EVX, Elanga, Energy Austra, Engie, Everty, Exploren, Fast Cities A, Gentari, JOLT, Noodoe, Porsche Destination Charging, Porsche Smart Mobility, Saascharge, Smart Charge, University of, Wevolt, Zeus Renewables


<a id="t2-type"></a>
## 5. Charger type and status

`Charger_Type` holds two different facts in one column: an electrical type (`AC`, `DC`) and, in 98
records, a lifecycle status (`Upcoming`). That is a modelling problem, not a formatting one. While
they share a column, no query can ask for "all DC chargers" without silently excluding every
planned DC site, and no query can ask "what is planned?" without pattern-matching on a type field.

Splitting them gives `charger_type` ∈ {AC, DC} and `charger_status` ∈ {Operational, Upcoming}. For
upcoming sites the type is genuinely unknown — the source does not record whether a planned site
will be AC or DC — so it is left missing rather than inferred from the power rating. A 350 kW
rating is strong evidence of DC, but *evidence* is not *data*, and Task 3 targets DC chargers
specifically: an inferred DC record would quietly enter that population as though it were observed.

In [7]:
# `Charger_Type` holds two different facts: the electrical type (AC/DC) and, for
# 98 records, a lifecycle status ('Upcoming'). Keeping them in one column means
# no query can ask for "all DC chargers" without silently excluding planned DC
# sites, so the two facts are separated into two fields.
def split_type_status(value):
    if pd.isna(value):
        return (pd.NA, pd.NA)
    text = str(value).strip().upper()
    if text in {"AC", "DC"}:
        return (text, "Operational")
    if text == "UPCOMING":
        # The source does not say whether a planned site will be AC or DC, so the
        # type is left missing rather than guessed from the power rating.
        return (pd.NA, "Upcoming")
    return (pd.NA, "Unknown")


df[["charger_type", "charger_status"]] = pd.DataFrame(
    df["charger_type_raw"].map(split_type_status).tolist(), index=df.index
)

record_step("type/status separated", "'Upcoming' moved into charger_status",
            int((df["charger_status"] == "Upcoming").sum()))
df.groupby(["charger_status", "charger_type"], dropna=False).size().to_frame("records")

  type/status separated          98  'Upcoming' moved into charger_status


records
charger_status charger_type         
Operational    AC               1427
               DC                433
Upcoming       NaN                98

<a id="t2-rating"></a>
## 6. Charger power ratings

`Charger_rating` mixes four formats across 46 distinct strings: `22 kW`, a bare `22`, the
non-numeric placeholder `AC` in 522 rows, and compound values such as `2x350kW & 2x175kW`.

The compound form is the interesting one, and it is why the parser returns a *list* of
`(connectors, kW)` pairs rather than a single number. `2x350kW & 2x175kW` describes two 350 kW
connectors and two 175 kW connectors at one site. Any single-number representation — maximum,
minimum, mean — throws away most of that. The list preserves it, and section 10 unpacks it into a
tidy table that Task 4's schema can store properly.

The placeholder `AC` is treated as missing rather than as a power of zero. It records a charger
*type* in a rating field, so the honest reading is that the rating is unknown.

This cell only defines the parser and profiles the formats. The derived columns come later, after
deduplication, so they describe the surviving record rather than one that was merged away.

In [8]:
RATING_SEPARATOR = re.compile(r"\s*(?:&|\+|,|/|\band\b)\s*", re.IGNORECASE)
RATING_TOKEN = re.compile(
    r"^(?:(?P<count>\d+)\s*x\s*)?(?P<kw>\d+(?:\.\d+)?)\s*(?:kw)?$", re.IGNORECASE
)


def parse_rating(value):
    """
    Turn one raw `Charger_rating` string into a list of (connectors, kW) pairs.

    Covers all four formats present in the source:
        '22 kW'              -> [(1, 22.0)]
        '22'                 -> [(1, 22.0)]
        '2x350kW & 2x175kW'  -> [(2, 350.0), (2, 175.0)]
        'AC'                 -> []            a placeholder, not a power value

    Returning a list rather than one number is the whole point: a compound value
    describes several connectors of different power at one site, and flattening
    it to a single figure destroys information the Task 4 schema needs.
    """
    if pd.isna(value):
        return []
    text = str(value).strip().lower().replace("\u00d7", "x")
    if not text:
        return []
    components = []
    for part in RATING_SEPARATOR.split(text):
        match = RATING_TOKEN.match(part.strip())
        if match:
            count = int(match.group("count")) if match.group("count") else 1
            components.append((count, float(match.group("kw"))))
    return components


def classify_rating(value) -> str:
    """Label which of the four formats a raw rating string uses."""
    text = "" if pd.isna(value) else str(value).strip()
    if re.fullmatch(r"\d+(\.\d+)?\s*kW", text, re.IGNORECASE):
        return "number+unit"
    if re.fullmatch(r"\d+(\.\d+)?", text):
        return "bare number"
    if re.search(r"\dx\s*\d", text, re.IGNORECASE):
        return "compound"
    if re.fullmatch(r"[A-Za-z ]+", text):
        return "non-numeric placeholder"
    return "other"


# Profiling only at this point. The derived numeric columns are computed after
# deduplication, so they are guaranteed to describe the record that survives.
df["charger_rating_raw"].map(classify_rating).value_counts().to_frame("records")

,records
charger_rating_raw,
number+unit,1315
non-numeric placeholder,522
compound,99
bare number,22


<a id="t2-address"></a>
## 7. Addresses and postcodes

The newlines are already gone; what remains is punctuation. Addresses arrive as `, Muswellbrook,
2333` (empty street line), `76 Wingewarra St, Dubbo , 2830` (space before the comma) and
`1 - 7 Ross St, Wilcannia NSW 2836, Australia`. Normalising the comma spacing and stripping the
stray leading and trailing separators gives one consistent single-line form.

The postcode needs a decision rather than a rule, because the dataset carries two of them and they
disagree. `PCODE` is null in 121 records, and demonstrably wrong in others: the Wilcannia record
has `PCODE = 2350` while both its address text and its coordinates place it in 2836. The postcode
embedded in the address is therefore preferred, `PCODE` fills the gaps, and every disagreement is
flagged rather than quietly resolved — the flag is what makes it possible to quantify how often
the two fields conflict.

The locality is left alone. 399 records give it as the placeholder `Sydney` when the true suburb is
somewhere else entirely, and there is no way to recover the real value from within the data. It is
flagged, and the SA4 join in section 12 supplies a trustworthy geography instead.

In [9]:
POSTCODE_IN_ADDRESS = re.compile(r"\b(\d{4})\b(?!.*\b\d{4}\b)")   # last 4-digit run


def in_nsw_postcode_range(code) -> bool:
    if pd.isna(code) or not str(code).isdigit():
        return False
    number = int(code)
    return any(low <= number <= high for low, high in NSW_POSTCODE_RANGES)


# The newlines are already gone (whitespace pass); what remains is punctuation
# spacing - ' ,', ',,' and stray leading/trailing commas from empty street lines.
df["station_address"] = (
    df["station_address_raw"]
    .str.replace(r"\s*,\s*", ", ", regex=True)
    .str.replace(r"(?:,\s*){2,}", ", ", regex=True)
    .str.replace(r"^[,\s]+", "", regex=True)
    .str.replace(r"[,\s]+$", "", regex=True)
)

df["postcode_from_address"] = df["station_address"].str.extract(POSTCODE_IN_ADDRESS, expand=False)
# The address is the more trustworthy of the two sources: PCODE is null in 121
# records and demonstrably wrong in others (the Wilcannia record carries 2350 but
# its address, and its coordinates, are in 2836).
df["postcode"] = df["postcode_from_address"].fillna(df["postcode_reported"])
df["postcode_conflict_flag"] = (
    df["postcode_from_address"].notna()
    & df["postcode_reported"].notna()
    & (df["postcode_from_address"] != df["postcode_reported"])
)
df["postcode_valid_flag"] = df["postcode"].map(in_nsw_postcode_range)

# The locality inside the address is a known-bad field: 399 records use 'Sydney'
# in place of the real suburb. It is flagged, not repaired - the SA4 join below
# supplies a trustworthy region instead.
df["locality_placeholder_flag"] = df["station_address"].str.contains(
    r",\s*Sydney\s*,", case=False, regex=True, na=False
)

record_step("addresses normalised", "addresses reduced to one clean line",
            int(df["station_address"].notna().sum()))
record_step("postcode recovered", "postcode recovered from the address text",
            int((df["postcode_reported"].isna() & df["postcode"].notna()).sum()))
record_step("postcode conflict", "PCODE disagrees with the address postcode",
            int(df["postcode_conflict_flag"].sum()))
record_step("postcode invalid", "postcode outside the NSW ranges",
            int((~df["postcode_valid_flag"]).sum()))
record_step("locality placeholder", "'Sydney' used in place of the real suburb",
            int(df["locality_placeholder_flag"].sum()))
df[["station_address", "postcode_reported", "postcode", "postcode_conflict_flag"]].head()

  addresses normalised        1,958  addresses reduced to one clean line
  postcode recovered            120  postcode recovered from the address text
  postcode conflict              35  PCODE disagrees with the address postcode
  postcode invalid                3  postcode outside the NSW ranges
  locality placeholder          399  'Sydney' used in place of the real suburb


,station_address,postcode_reported,postcode,postcode_conflict_flag
0,"Muswellbrook, 2333",2333,2333,False
1,"01 Wallgrove Road, Sydney, 2766",2766,2766,False
2,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",2350,2836,True
3,"1 Balfour St, Sydney, 2070",2070,2070,False
4,"1 Bay Ln, Byron Bay, 2481",2481,2481,False


<a id="t2-coords"></a>
## 8. Coordinates and sites

Coordinates were rounded to six decimal places in section 3.3 — about 11 cm, already finer than
the source can justify. That rounding matters here: unrounded floating-point values make two
records at one physical site compare as different locations, which would defeat the deduplication
below.

`site_id` groups records sharing a location. It exists because "duplicate coordinates" is
ambiguous in this dataset. Task 1 (section 6.5) found 20 records sharing a location with another, but they
are not all duplicates: at 19 Princes Hwy, Figtree, a Tesla DC unit and a non-networked AC unit
genuinely coexist, while the same Tesla charger also appears twice because two feeds describe it
differently. Separating *site* from *charger* lets the next cell treat those two situations
differently instead of applying one rule to both and getting one of them wrong.

In [10]:
missing_coords = df["latitude"].isna() | df["longitude"].isna()
outside_nsw = ~(
    df["latitude"].between(*NSW_LAT_RANGE) & df["longitude"].between(*NSW_LON_RANGE)
)
df["coordinate_valid_flag"] = ~(missing_coords | outside_nsw)

record_step("coordinates missing", "records with no usable coordinates", int(missing_coords.sum()))
record_step("coordinates off-NSW", "records outside the NSW bounding box",
            int((outside_nsw & ~missing_coords).sum()))

# `site_id` groups records that share a physical location. It is what makes the
# difference between a duplicate (same site, same charger, two feeds) and a
# legitimate co-located pair (same site, one AC unit and one DC unit) something
# the data can express, instead of something dedup has to guess at.
site_key = (
    df["latitude"].round(COORDINATE_DECIMALS).astype("string")
    + "," + df["longitude"].round(COORDINATE_DECIMALS).astype("string")
)
df["site_id"] = pd.factorize(site_key)[0] + 1
print(f"{df['site_id'].nunique():,} distinct sites across {len(df):,} records")

  coordinates missing             0  records with no usable coordinates
  coordinates off-NSW             0  records outside the NSW bounding box
1,935 distinct sites across 1,958 records


<a id="t2-dupes"></a>
## 9. Duplicates and reconciliation

Two passes handle duplicates *at the same site*; a third, in section 9.1, looks for the same
charger recorded at two slightly different sites.

**Exact duplicates** are identical on every field that identifies a charger. One copy is redundant
and is dropped.

**Conflicting duplicates** are the same charger at the same site, described differently by two
source feeds. At 520 David St, Albury, one row reads `AC` / 4 plugs and the other `7` / 2 plugs.
Dropping either loses information the other one has, so the group is merged under rules stated per
field:

| Field | Rule | Why |
|---|---|---|
| rating | prefer a value that parses; then the one describing most connectors | `'7'` beats the placeholder `'AC'`; `'2x350kW & 2x175kW'` beats `'175 kW'` |
| plugs | the larger count | a feed reporting fewer is reporting a subset of the bays |
| text fields | the longest non-null value | the least truncated version of the same string |
| flags | logical OR | a problem noted on either row survives the merge |

The grouping key is `(site_id, operator, charger_type, charger_status)` — and it only works because
operator canonicalisation ran first. Before section 4, the Figtree pair reads `Tesla` and
`Tesla Motors ` and looks like two different chargers.

`charger_id` is assigned last, once the record set is final. It is the primary key in Task 4 and
the join target for Task 3's augmented attributes, so it must be stable — assigning it before
deduplication would leave gaps and shift meaning between runs.

In [11]:
# Two different problems, handled in two passes.
#
# Pass 1 - exact duplicates: identical on every field that identifies a charger.
# One copy is simply redundant.
IDENTITY_COLUMNS = [
    "site_id", "operator", "charger_type", "charger_status",
    "charger_rating_raw", "number_of_plugs", "station_address",
]
exact_duplicates = df.duplicated(subset=IDENTITY_COLUMNS, keep="first")
df = df.loc[~exact_duplicates].copy()
record_step("exact duplicates", "identical records removed", int(exact_duplicates.sum()))

# Pass 2 - conflicting duplicates: the same charger at the same site described
# differently by two feeds (rating 'AC' with 4 plugs in one row, '7' with 2 plugs
# in the other). Dropping either row loses real information, so the group is
# merged under rules chosen per field:
#   rating  - prefer the value that actually parses to a power figure, then the
#             one describing the most connectors; the 'AC' placeholder loses
#   plugs   - the larger count; a feed reporting fewer is reporting a subset
#   text    - the longest non-null value, i.e. the least truncated
#   flags   - OR'd, so a problem noted on either row survives the merge
GROUP_KEY = ["site_id", "operator", "charger_type", "charger_status"]
conflicting = df.duplicated(subset=GROUP_KEY, keep=False)
conflict_groups = int(df.loc[conflicting, GROUP_KEY].drop_duplicates().shape[0])
print(f"{int(conflicting.sum())} record(s) in {conflict_groups} conflicting group(s)")


def longest(series):
    """The longest non-null string in the group - the least truncated one."""
    values = series.dropna().astype(str)
    return max(values, key=len) if len(values) else pd.NA


def best_rating(series):
    """
    The most informative rating in the group.

    Sort key: parses at all > describes more connectors > longer string. This is
    what keeps '7' rather than the placeholder 'AC', and keeps
    '2x350kW & 2x175kW' rather than the single-figure '175 kW'.
    """
    values = series.dropna().astype(str)
    if not len(values):
        return pd.NA
    return max(values, key=lambda v: (len(parse_rating(v)) > 0, len(parse_rating(v)), len(v)))


AGGREGATIONS = {
    "source_object_id": "first",
    "station_name": longest,
    "station_address": longest,
    "station_address_raw": longest,
    "operator_raw": longest,
    "charger_type_raw": "first",
    "charger_rating_raw": best_rating,
    "number_of_plugs": "max",
    "latitude": "first",
    "longitude": "first",
    "lga_name": longest,
    "postcode": longest,
    "postcode_reported": longest,
    "postcode_from_address": longest,
    "source_feed": longest,
    "operator_truncated_flag": "max",
    "postcode_conflict_flag": "max",
    "postcode_valid_flag": "max",
    "locality_placeholder_flag": "max",
    "coordinate_valid_flag": "min",
}

before_rows = len(df)
df = df.groupby(GROUP_KEY, dropna=False, as_index=False).agg(AGGREGATIONS)
record_step("duplicates reconciled", "conflicting records merged into one",
            before_rows - len(df))

# A stable surrogate key, assigned once the record set is final. Task 4 uses it
# as the primary key; Task 3 uses it as the join target for augmented attributes.
df = df.sort_values(["site_id", "operator", "charger_type"]).reset_index(drop=True)
df.insert(0, "charger_id", range(1, len(df) + 1))
print(f"{before_rows:,} -> {len(df):,} records after reconciliation")

  exact duplicates                0  identical records removed
24 record(s) in 12 conflicting group(s)


  duplicates reconciled          12  conflicting records merged into one
1,958 -> 1,946 records after reconciliation


### 9.1 The same charger in two feeds

The two passes above compare only records at the same site, and a site is an exact coordinate
(section 8). That misses a third kind of duplicate. TfNSW compiles this dataset from several
programme feeds (`Existing Destination Chargers`, `Destination Charging R2`, `Kerbside Charging R1`
and others), and some chargers appear in two of them: with coordinates a fraction of a metre apart,
or with the same street address geocoded a few metres apart. `179 Gillards Rd, Pokolbin, 2320` and
`179 Gillards Rd Pokolbin NSW 2320 Australia` are 0.1 m apart, with the same operator, type and plug
count, from different feeds.

A pair of records is flagged as a **probable duplicate** when all of the following hold:

* the same operator, charger type and status;
* **different source feeds**. Within one feed, two records at one address are more likely two
  separate units (there are two JOLT chargers at 15 Merriville Rd, for example), so same-feed pairs
  are left alone;
* and either the two points are within **2 m** of each other (the same coordinate, to GPS
  precision), or they share the **same street number and street name** and are within **50 m**.

The rule is deliberately conservative, and the pairs are **flagged, not merged**. The evidence is
strong but not conclusive: two records at one street address could still be two chargers, and this
notebook repairs only where a rule can be justified. There is also a practical reason. Merging would
renumber `charger_id`, which Task 3's cached API searches are keyed on.

In each pair, the more complete record is kept as the representative: the one whose address has a
street number, then the one with more populated fields, then the lower `charger_id`. That is the
preference Task 3 uses for shared OCM sites. The other record gets `probable_duplicate_flag = True`,
so an analysis that needs one row per physical charger can exclude it. The pair evidence goes to
`probable_duplicates.csv`, and Task 4 stores it as a link table.

In [12]:
# Pass 3 - the same charger in two source feeds. Passes 1 and 2 compare only
# records at one site (identical coordinates), so a charger that two feeds
# report a fraction of a metre apart survives both. Pairs are flagged, not
# merged: the evidence is strong but not conclusive, and merging would
# renumber charger_id, which Task 3's cached API searches are keyed on.
SAME_POINT_M = 2              # within GPS precision: one coordinate written twice
SAME_ADDRESS_RADIUS_M = 50    # a shared street address counts only this close

STREET_SUFFIXES = {
    "street": "st", "road": "rd", "avenue": "ave", "highway": "hwy", "drive": "dr",
    "parade": "pde", "lane": "ln", "place": "pl", "crescent": "cres", "court": "ct",
    "terrace": "tce", "boulevard": "blvd", "circuit": "cct", "close": "cl",
    "esplanade": "esp", "square": "sq", "grove": "gr",
}
STREET_ADDRESS = re.compile(
    r"^(?P<number>\d+[a-z]?(?:-\d+[a-z]?)?) (?P<street>[a-z ]+? (?:"
    + "|".join(sorted(set(STREET_SUFFIXES.values()) | {"way", "row", "mews"}))
    + r"))\b"
)


def street_key(address):
    """
    Reduce an address to 'number street', e.g. '106 parraween st', or None.

    Only the house number and street are compared, because the locality is the
    unreliable part of these addresses (the 'Sydney' placeholder, section 7).
    An address with no house number gives no key: 'Gardeners Rd' alone is too
    vague to say that two records describe the same charger.
    """
    if pd.isna(address):
        return None
    text = str(address).lower().replace("/", "-")
    text = re.sub(r"\s*-\s*", "-", text)                 # '2 - 8' -> '2-8'
    text = re.sub(r"[^a-z0-9\- ]", " ", text)
    for long_form, short_form in STREET_SUFFIXES.items():
        text = re.sub(rf"\b{long_form}\b", short_form, text)
    match = STREET_ADDRESS.match(re.sub(r"\s+", " ", text).strip())
    return f"{match['number']} {match['street']}" if match else None


def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres; ample precision at a 50 m scale."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * 6_371_008.8 * np.arcsin(np.sqrt(a))


UNIT_KEY = ["operator", "charger_type", "charger_status"]
records = df[["charger_id", *UNIT_KEY, "source_feed", "latitude", "longitude",
              "station_address"]].copy()
records["charger_type"] = records["charger_type"].fillna("unknown")   # Upcoming sites
records["feed"] = records["source_feed"].fillna("(not recorded)")
records["street_key"] = records["station_address"].map(street_key)

# Candidate pairs share operator, type and status: a self-join on those keys.
pairs = records.merge(records, on=UNIT_KEY, suffixes=("_a", "_b"))
pairs = pairs[pairs["charger_id_a"] < pairs["charger_id_b"]].copy()
pairs["distance_m"] = haversine_m(pairs["latitude_a"], pairs["longitude_a"],
                                  pairs["latitude_b"], pairs["longitude_b"])
same_point = pairs["distance_m"] <= SAME_POINT_M
same_address = (pairs["street_key_a"].notna()
                & (pairs["street_key_a"] == pairs["street_key_b"])
                & (pairs["distance_m"] <= SAME_ADDRESS_RADIUS_M))
pairs["rule"] = np.select([same_point, same_address], ["same point", "same street address"],
                          default="")
duplicate_pairs = pairs[(pairs["feed_a"] != pairs["feed_b"]) & (pairs["rule"] != "")].copy()

# In this release every record is in at most one pair, so each pair simply names
# a representative. A future release that produces chains (a~b~c) stops here for
# review, rather than being resolved by a rule nobody has examined.
in_pairs = pd.concat([duplicate_pairs["charger_id_a"], duplicate_pairs["charger_id_b"]])
assert in_pairs.is_unique, "a record belongs to two duplicate pairs - review before flagging"

indexed = df.set_index("charger_id")
completeness = pd.DataFrame({
    "has_number": indexed["station_address"].str.match(r"\d", na=False),
    "populated": indexed.notna().sum(axis=1),
})


def representative(a, b):
    """The more complete of two records: street number, then filled fields, then lower id."""
    score = lambda cid: (completeness.at[cid, "has_number"], completeness.at[cid, "populated"], -cid)
    return a if score(a) >= score(b) else b


duplicate_pairs["duplicate_of"] = [
    representative(a, b) for a, b in zip(duplicate_pairs["charger_id_a"], duplicate_pairs["charger_id_b"])
]
duplicate_pairs["charger_id"] = np.where(
    duplicate_pairs["duplicate_of"] == duplicate_pairs["charger_id_a"],
    duplicate_pairs["charger_id_b"], duplicate_pairs["charger_id_a"],
)
probable_duplicates = (
    duplicate_pairs[["charger_id", "duplicate_of", "rule", "distance_m"]]
    .round({"distance_m": 1})
    .sort_values("charger_id")
    .reset_index(drop=True)
)
df["probable_duplicate_flag"] = df["charger_id"].isin(probable_duplicates["charger_id"])

record_step("probable duplicates", "same charger in two feeds: flagged, not merged",
            len(probable_duplicates))
display(
    probable_duplicates
    .assign(operator=lambda t: t["charger_id"].map(indexed["operator"]),
            type=lambda t: t["charger_id"].map(indexed["charger_type"]).fillna("Upcoming"),
            flagged_address=lambda t: t["charger_id"].map(indexed["station_address"]),
            kept_address=lambda t: t["duplicate_of"].map(indexed["station_address"]),
            flagged_feed=lambda t: t["charger_id"].map(indexed["source_feed"]),
            kept_feed=lambda t: t["duplicate_of"].map(indexed["source_feed"]))
    .sort_values(["rule", "distance_m"])
    .reset_index(drop=True)
)

  probable duplicates            23  same charger in two feeds: flagged, not merged


,charger_id,duplicate_of,rule,distance_m,operator,type,flagged_address,kept_address,flagged_feed,kept_feed
0,222,1302,same point,0.1,Exploren,AC,"179 Gillards Rd, Pokolbin, 2320",179 Gillards Rd Pokolbin NSW 2320 Australia,Existing Destination Chargers,Destination Charging R2
1,550,559,same point,0.1,Chargefox,AC,"43 Station St, Newcastle, 2293","44 Station St, Wickham NSW 2293, Australia",Existing Destination Chargers,Kerbside Charging R1
2,560,608,same point,0.1,Exploren,AC,"446 Dean St, Albury, 2640","520 David St, Albury NSW 2640, Australia",Existing Destination Chargers,Destination Charging R1
3,602,1580,same point,0.1,Exploren,AC,"51 Bathurst St, Condobolin, 2877",51 Bathurst St Condobolin NSW 2877 Australia,Existing Destination Chargers,Destination Charging R2
4,820,243,same point,0.1,Exploren,AC,"Buchanan Dr, South West Rocks NSW 2431, Australia","19 Buchanan Drive, South West Rocks, 2431",Destination Charging R1,Existing Destination Chargers
5,821,1691,same point,0.1,Chargefox,DC,"Bunnerong Rd, Sydney, 2036",801-899R Bunnerong Rd Chifley NSW 2036 Australia,Existing Fast Chargers,Kerbside Charging R1
6,909,1552,same point,0.1,Exploren,AC,"Merry Beach Rd, Kioloa, 2539",46 Merry Beach Rd Kioloa NSW 2539 Australia,Existing Destination Chargers,Destination Charging R2
7,940,1391,same point,0.1,Exploren,AC,"Princes Hwy, Ulladulla, 2539",222 Princes Hwy Ulladulla NSW 2539 Australia,Existing Destination Chargers,Destination Charging R2
8,967,1796,same point,0.1,Chargefox,AC,"Tathra Bermagui Rd, Tathra, 2550",1 Andy Poole Dr Tathra NSW 2550 Australia,Existing Destination Chargers,Destination Charging R2
9,1555,577,same point,0.1,Chargefox,DC,47-49A Cleary St Hamilton NSW 2303 Australia,"47-49A Cleary St, Newcastle, 2303",Kerbside Charging R1,Existing Fast Chargers


<a id="t2-ratingfields"></a>
## 10. Derived rating fields

Now that each charger is one record, the parser from section 6 is applied to produce
`power_kw_min`, `power_kw_max` and a tidy component table.

One subtlety in `connectors_in_rating`: only a compound rating states how many connectors exist.
`22 kW` states the power *per* connector and says nothing at all about their number, so counting it
as one connector would invent a fact — and would make the plug-count cross-check in section 11 fire
on almost every row, drowning the real signal in noise. The column is therefore left missing except
where the rating explicitly counts connectors.

The component table is what makes the compound ratings queryable. `power_kw_max` answers "how fast
is this charger?"; `charger_power_ratings.csv` answers "what is actually installed there?" — a
question a single column cannot hold an answer to, and one that Task 4's normalised schema is
designed for.

In [13]:
# The derived numeric fields are computed here, after reconciliation, so they
# always describe the rating string that actually survived the merge.
parsed = df["charger_rating_raw"].map(parse_rating)
df["rating_format"] = df["charger_rating_raw"].map(classify_rating)
df["power_kw_max"] = parsed.map(lambda comps: max(kw for _, kw in comps) if comps else np.nan)
df["power_kw_min"] = parsed.map(lambda comps: min(kw for _, kw in comps) if comps else np.nan)

# Only a compound rating ('2x350kW') states how many connectors exist. A plain
# '22 kW' states the power *per* connector and says nothing about their number,
# so treating it as one connector would invent a fact and make the plug-count
# cross-check below fire on almost every row.
df["connectors_in_rating"] = (
    parsed.map(lambda comps: sum(count for count, _ in comps) if comps else pd.NA)
          .where(df["rating_format"] == "compound", pd.NA)
          .astype("Int64")
)

record_step("rating parsed", "ratings resolved to a numeric kW value",
            int(df["power_kw_max"].notna().sum()))
record_step("rating unparsable", "placeholder ratings left as NA (e.g. 'AC')",
            int(df["power_kw_max"].isna().sum()))

# The compound ratings are also emitted as a tidy one-row-per-component table.
# `power_kw_max` answers "how fast is this charger?"; this answers "what is
# actually installed there?", which is what a normalised schema needs and what a
# single column cannot hold.
power_ratings = pd.DataFrame(
    [
        {"charger_id": charger_id, "component_no": position,
         "connector_count": count, "power_kw": kw}
        for charger_id, components in zip(df["charger_id"], parsed)
        for position, (count, kw) in enumerate(components, start=1)
    ],
    columns=["charger_id", "component_no", "connector_count", "power_kw"],
)
print(f"{len(power_ratings):,} rating components for "
      f"{power_ratings['charger_id'].nunique():,} chargers")
df["rating_format"].value_counts().to_frame("records")

  rating parsed               1,428  ratings resolved to a numeric kW value
  rating unparsable             518  placeholder ratings left as NA (e.g. 'AC')
1,527 rating components for 1,428 chargers


,records
rating_format,
number+unit,1314
non-numeric placeholder,518
compound,99
bare number,15


<a id="t2-consistency"></a>
## 11. Cross-field consistency

These checks change no values. They mark records whose own fields contradict each other, which is
the "inconsistent charger attributes" the brief asks about — inconsistency *within* a record, not
just between records.

Three checks run. A `DC` charger whose rating reads `AC` cannot be both. A compound rating counting
four connectors on a record reporting six plugs disagrees with itself. And a missing station name,
while not a contradiction, matters enough to Task 3 to be worth a flag: with 432 of 433 DC chargers
unnamed, name-based matching against an external API is not viable and coordinates are the only
usable join key.

In [14]:
# Cross-field checks. These change no values; they mark records whose own fields
# contradict each other, so consistency can be quantified and Task 3 can
# avoid augmenting a record that is internally unreliable.
df["type_rating_conflict_flag"] = (
    (df["charger_type"] == "DC")
    & df["charger_rating_raw"].str.upper().eq("AC").fillna(False)
)
df["plug_count_conflict_flag"] = (
    df["connectors_in_rating"].notna()
    & df["number_of_plugs"].notna()
    & (df["connectors_in_rating"] != df["number_of_plugs"])
)
df["missing_name_flag"] = df["station_name"].isna()

record_step("type vs rating", "DC records whose rating reads 'AC'",
            int(df["type_rating_conflict_flag"].sum()))
record_step("plugs vs connectors", "plug count disagrees with the parsed rating",
            int(df["plug_count_conflict_flag"].sum()))
record_step("station name missing", "records with no station name",
            int(df["missing_name_flag"].sum()))

  type vs rating                  0  DC records whose rating reads 'AC'
  plugs vs connectors            36  plug count disagrees with the parsed rating
  station name missing        1,430  records with no station name


<a id="t2-spatial"></a>
## 12. Spatial integration with ASGS SA4

This is the join the brief asks for: add a field to each EV charger naming the SA4 region it falls
in. It runs in DuckDB with the `spatial` extension, for the reason given in Task 1, section 7 — Task 4 has
to store this data in DuckDB anyway, so doing the geometry there keeps one engine rather than
moving data between GeoPandas and a database.

### 12.1 Reconciling the coordinate reference systems

The two datasets do not share a CRS. The ABS boundaries are GDA2020 (EPSG:7844); the TfNSW
latitude and longitude columns are WGS84 (EPSG:4326). At the current epoch the two differ by under
about 2 m, which cannot move a charger across an SA4 boundary that is kilometres wide — but
"close enough" is not a decision worth leaving implicit in a spatial join, so the polygons are
reprojected explicitly and the choice is recorded in the ledger.

`always_xy := true` is the part that actually bites. EPSG:4326 formally orders its axes
latitude-then-longitude, while `ST_Point` takes `(x, y)` — longitude first. Without that flag the
transform returns coordinates in the opposite order from the points being tested against them, and
every charger silently matches nothing.

In [15]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# The boundary file is GDA2020 (EPSG:7844); the TfNSW coordinates are WGS84
# (EPSG:4326). The two are close - under about 2 m at the current epoch - but
# "close enough" is not a decision worth leaving implicit in a spatial join, so
# the polygons are reprojected explicitly.
#
# `always_xy := true` is the part that matters. EPSG:4326 formally orders its
# axes latitude-then-longitude, so without it the transform returns coordinates
# in the opposite order from ST_Point(longitude, latitude) and every point lands
# in the ocean off East Africa. If the local PROJ build cannot perform the
# transform, the untransformed geometry is used and the fallback is logged: a
# sub-2 m offset cannot move a charger across an SA4 boundary kilometres wide.
TRANSFORMED = f"ST_Transform(geom, '{SOURCE_CRS}', '{TARGET_CRS}', always_xy := true)"
SA4_COLUMNS = """
    SA4_CODE26 AS sa4_code,
    SA4_NAME26 AS sa4_name,
    GCC_CODE26 AS gcc_code,
    GCC_NAME26 AS gcc_name,
    STE_CODE26 AS state_code,
    STE_NAME26 AS state_name,
    AREASQKM26 AS area_sqkm
"""
try:
    con.execute(f"""
        CREATE OR REPLACE TABLE sa4 AS
        SELECT {SA4_COLUMNS}, {TRANSFORMED} AS geom
        FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')
    """)
    crs_note = f"reprojected {SOURCE_CRS} -> {TARGET_CRS} (always_xy)"
except duckdb.Error as error:
    print(f"WARNING: ST_Transform unavailable ({str(error)[:120]}); "
          "using GDA2020 coordinates directly (offset < 2 m).")
    con.execute(f"""
        CREATE OR REPLACE TABLE sa4 AS
        SELECT {SA4_COLUMNS}, geom
        FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')
    """)
    crs_note = f"{SOURCE_CRS} treated as {TARGET_CRS} (offset < 2 m)"

record_step("CRS reconciled", crs_note,
            int(con.execute("SELECT COUNT(*) FROM sa4").fetchone()[0]))
con.execute("SELECT state_name, COUNT(*) AS regions FROM sa4 GROUP BY 1 ORDER BY 2 DESC").df()

  CRS reconciled                108  reprojected EPSG:7844 -> EPSG:4326 (always_xy)


,state_name,regions
0,New South Wales,30
1,Queensland,21
2,Victoria,19
3,Western Australia,12
4,South Australia,9
5,Tasmania,6
6,Northern Territory,4
7,Other Territories,3
8,Australian Capital Territory,3
9,Outside Australia,1


### 12.2 The point-in-polygon join

Each charger becomes a point and is tested for containment in each SA4 polygon.

The join runs against all 108 Australian SA4s rather than a pre-filtered NSW subset, which is a
deliberate choice. Filtering to NSW first would make an interstate charger — a genuine data error
worth reporting — come back unmatched and indistinguishable from a geometry failure. Joining
against the whole country means the result *tells* us which state each point is in, and section 13
asserts that every one of them is NSW.

A `LEFT JOIN` rather than an inner join, so that an unmatched charger survives as a row with a null
region instead of vanishing from the dataset.

In [16]:
# The cleaned chargers are handed to DuckDB as a view over the DataFrame - no
# copy, no intermediate file.
chargers_for_join = df[["charger_id", "longitude", "latitude"]].astype(
    {"charger_id": "int64", "longitude": "float64", "latitude": "float64"}
)
con.register("chargers_py", chargers_for_join)

# ST_Point takes (x, y) = (longitude, latitude). Passing them the other way round
# is the most common error in this kind of join and it fails silently - every
# point simply matches nothing.
#
# The join runs against all 108 Australian SA4s rather than a pre-filtered NSW
# subset. Filtering first would hide a genuine problem: a charger whose
# coordinates put it outside NSW would come back unmatched and look like a
# geometry failure, instead of being correctly reported as an interstate point.
con.execute("""
    CREATE OR REPLACE TABLE charger_sa4 AS
    SELECT c.charger_id,
           s.sa4_code, s.sa4_name, s.gcc_code, s.gcc_name, s.state_name,
           CASE WHEN s.sa4_code IS NULL THEN NULL ELSE 'point-in-polygon' END
               AS sa4_match_method
    FROM chargers_py AS c
    LEFT JOIN sa4 AS s
      ON ST_Contains(s.geom, ST_Point(c.longitude, c.latitude))
""")

matched = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NOT NULL"
).fetchone()[0]
print(f"{matched:,} of {len(df):,} chargers matched by point-in-polygon "
      f"({matched / len(df) * 100:.1f}%)")

1,945 of 1,946 chargers matched by point-in-polygon (99.9%)


### 12.3 Chargers that fall outside every polygon

A charger on a jetty, a reclaimed wharf or a coastal car park can sit a few metres outside the
coastline the ABS digitised, and point-in-polygon returns nothing for it. Dropping those records
would bias the coverage analysis toward inland regions; leaving the field null would push the
problem into Task 4.

Instead each unmatched charger is assigned the nearest SA4, but only within a stated tolerance of
about 2 km, and `sa4_match_method` records which chargers were assigned this way. That last part is
what keeps the fallback transparent: it shows exactly how many chargers were assigned exactly and
how many were assisted, rather than presenting both as the same kind of result.

`ST_Distance` on geographic coordinates returns degrees, not metres, which is why the tolerance is
expressed in degrees. At NSW latitudes 0.02° is roughly 2 km — precise enough for a sanity
threshold, and the exact figure does not matter as long as it is small.

In [17]:
# A charger on a jetty, a reclaimed wharf or a coastal car park can sit a few
# metres outside the coastline the ABS digitised, and point-in-polygon returns
# nothing for it. Rather than dropping those records or leaving the field null,
# each is assigned the nearest SA4 - but only within a stated tolerance, and the
# method is recorded in `sa4_match_method` so exact matches can be told
# apart from assisted ones.
NEAREST_TOLERANCE_DEGREES = 0.02       # ~2 km at NSW latitudes

unmatched_count = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NULL"
).fetchone()[0]

if unmatched_count:
    con.execute(f"""
        CREATE OR REPLACE TABLE charger_sa4_nearest AS
        WITH candidates AS (
            SELECT u.charger_id, s.sa4_code, s.sa4_name, s.gcc_code, s.gcc_name,
                   s.state_name,
                   ST_Distance(s.geom, ST_Point(u.longitude, u.latitude)) AS distance_deg,
                   ROW_NUMBER() OVER (
                       PARTITION BY u.charger_id
                       ORDER BY ST_Distance(s.geom, ST_Point(u.longitude, u.latitude))
                   ) AS rank
            FROM chargers_py AS u
            JOIN charger_sa4 AS m
              ON m.charger_id = u.charger_id AND m.sa4_code IS NULL
            CROSS JOIN sa4 AS s
            WHERE NOT ST_IsEmpty(s.geom)
        )
        SELECT charger_id, sa4_code, sa4_name, gcc_code, gcc_name,
               state_name, distance_deg
        FROM candidates
        WHERE rank = 1 AND distance_deg <= {NEAREST_TOLERANCE_DEGREES}
    """)
    con.execute("""
        UPDATE charger_sa4 AS t
        SET sa4_code = n.sa4_code, sa4_name = n.sa4_name,
            gcc_code = n.gcc_code, gcc_name = n.gcc_name,
            state_name = n.state_name, sa4_match_method = 'nearest-polygon'
        FROM charger_sa4_nearest AS n
        WHERE t.charger_id = n.charger_id
    """)
    rescued = con.execute(
        "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_match_method = 'nearest-polygon'"
    ).fetchone()[0]
    print(f"{rescued} of {unmatched_count} unmatched charger(s) assigned by nearest polygon")
else:
    print("Every charger fell inside an SA4 polygon; no fallback needed.")

still_unmatched = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NULL"
).fetchone()[0]
record_step("SA4 assigned", "chargers given an SA4 region", len(df) - still_unmatched)
record_step("SA4 unassigned", "chargers left without an SA4 region", still_unmatched)

1 of 1 unmatched charger(s) assigned by nearest polygon
  SA4 assigned                1,946  chargers given an SA4 region
  SA4 unassigned                  0  chargers left without an SA4 region


### 12.4 Merging the regions back

The SA4 fields join back onto the cleaned DataFrame on `charger_id`. `validate="one_to_one"` makes
pandas raise if the join is not one-to-one, catching a duplicated key immediately rather than
letting it silently inflate the row count.

Two of the 30 NSW codes are ABS *non-spatial* codes, `Migratory - Offshore - Shipping (NSW)` and
`No usual address (NSW)`. They have no boundary, so no charger can ever lie in them. They are kept,
since they are part of the ASGS, but marked `has_boundary = False`, so that nothing downstream
mistakes them for regions without coverage.

The NSW SA4 table is also kept as an output in its own right. It becomes the region dimension in
Task 4, and it is the only thing that can show a region with **no** chargers at all — a join result
can only ever list regions that already have one, and for a coverage analysis the empty regions are
the interesting ones.

In [18]:
sa4_lookup = con.execute("""
    SELECT charger_id, sa4_code, sa4_name, gcc_code, gcc_name,
           state_name, sa4_match_method
    FROM charger_sa4
""").df()

df = df.merge(sa4_lookup, on="charger_id", how="left", validate="one_to_one")

# The NSW SA4 reference table becomes its own output: Task 4 stores it as the
# region dimension, and it is what makes a region with *no* chargers visible.
# A join result alone can only ever show regions that already have one.
sa4_regions_nsw = con.execute("""
    SELECT sa4_code, sa4_name, gcc_code, gcc_name, state_name,
           ROUND(area_sqkm, 1) AS area_sqkm,
           geom IS NOT NULL AS has_boundary    -- FALSE for the ABS non-spatial codes
    FROM sa4
    WHERE state_name = 'New South Wales'
    ORDER BY sa4_code
""").df()

with_boundary = sa4_regions_nsw["has_boundary"]
print(f"{len(sa4_regions_nsw)} NSW SA4 codes: {with_boundary.sum()} regions with a boundary and "
      f"{(~with_boundary).sum()} ABS non-spatial codes "
      f"({', '.join(sa4_regions_nsw.loc[~with_boundary, 'sa4_name'])})")
print(f"{df['sa4_code'].nunique()} of the {with_boundary.sum()} regions contain at least one charger")
df[["charger_id", "operator", "charger_type", "latitude", "longitude",
    "sa4_code", "sa4_name", "sa4_match_method"]].head()

30 NSW SA4 codes: 28 regions with a boundary and 2 ABS non-spatial codes (Migratory - Offshore - Shipping (NSW), No usual address (NSW))
28 of the 28 regions contain at least one charger


,charger_id,operator,charger_type,latitude,longitude,sa4_code,sa4_name,sa4_match_method
0,1,EVUp,AC,-32.262242,150.890139,106,Hunter Valley exc Newcastle,point-in-polygon
1,2,BP Australia,DC,-33.811004,150.849597,116,Sydney - Blacktown,point-in-polygon
2,3,NRMA,DC,-30.511874,151.669395,110,New England and North West,point-in-polygon
3,4,Chargefox,AC,-33.774101,151.167035,121,Sydney - North Sydney and Hornsby,point-in-polygon
4,5,Tesla,AC,-28.641819,153.613633,112,Richmond - Tweed,point-in-polygon


<a id="t2-validate"></a>
## 13. Validation and summary

Each check below is an assumption the rest of the pipeline relies on: that `charger_id` is unique,
that `charger_type` holds nothing but AC and DC, that every referenced charger in the ratings table
exists, that no charger landed in another state. Stating them as checks rather than eyeballing the
output means a future TfNSW release that breaks one stops the notebook here, instead of surfacing
as a wrong number in a Task 4 query.

In [19]:
# Post-conditions. Each is an assumption the rest of the pipeline relies on, so
# each is asserted rather than eyeballed: a future TfNSW release that breaks one
# should stop the notebook here, not surface as a wrong number in Task 4.
problems = []

if df["charger_id"].duplicated().any():
    problems.append("charger_id is not unique")
if df["charger_id"].isna().any():
    problems.append("charger_id contains nulls")
if not set(df["charger_type"].dropna()) <= {"AC", "DC"}:
    problems.append(f"unexpected charger_type values: {set(df['charger_type'].dropna())}")
if not set(df["charger_status"].dropna()) <= {"Operational", "Upcoming", "Unknown"}:
    problems.append("unexpected charger_status values")
if not bool(df["latitude"].dropna().between(*NSW_LAT_RANGE).all()):
    problems.append("latitude outside the NSW range")
if not bool(power_ratings["charger_id"].isin(df["charger_id"]).all()):
    problems.append("power_ratings references an unknown charger_id")

duplicate_targets = probable_duplicates["duplicate_of"]
if not duplicate_targets.isin(df["charger_id"]).all():
    problems.append("a probable duplicate points to an unknown charger_id")
if duplicate_targets.isin(probable_duplicates["charger_id"]).any():
    problems.append("a representative record is itself flagged as a duplicate")

off_state = df.loc[df["state_name"].notna() & (df["state_name"] != "New South Wales")]
if len(off_state):
    problems.append(f"{len(off_state)} charger(s) matched an SA4 outside NSW")

print("Validation:", "PASSED" if not problems else "FAILED")
for problem in problems:
    print("  -", problem)

summary = pd.DataFrame({
    "metric": [
        "raw records", "clean records", "distinct sites", "distinct operators",
        "DC chargers", "AC chargers", "upcoming sites",
        "probable duplicates (flagged, not merged)",
        "records with an SA4", "NSW SA4 regions with a boundary",
        "NSW SA4 regions with no charger", "ABS non-spatial SA4 codes (no boundary)",
    ],
    "value": [
        len(raw), len(df), df["site_id"].nunique(), df["operator"].nunique(),
        int((df["charger_type"] == "DC").sum()), int((df["charger_type"] == "AC").sum()),
        int((df["charger_status"] == "Upcoming").sum()),
        int(df["probable_duplicate_flag"].sum()),
        int(df["sa4_code"].notna().sum()), int(with_boundary.sum()),
        int((~sa4_regions_nsw.loc[with_boundary, "sa4_code"].isin(df["sa4_code"])).sum()),
        int((~with_boundary).sum()),
    ],
})
summary

Validation: PASSED


,metric,value
0,raw records,1958
1,clean records,1946
2,distinct sites,1935
3,distinct operators,42
4,DC chargers,431
5,AC chargers,1417
6,upcoming sites,98
7,"probable duplicates (flagged, not merged)",23
8,records with an SA4,1946
9,NSW SA4 regions with a boundary,28


The per-region distribution below is the first thing the cleaned data makes possible. It is built
from the full SA4 list rather than from the join result,
so a region with zero chargers would appear as a zero instead of disappearing. In this release
every region has chargers. The two non-spatial codes are left out, because they are not places.

In [20]:
chargers_by_sa4 = (
    sa4_regions_nsw.loc[sa4_regions_nsw["has_boundary"], ["sa4_code", "sa4_name", "gcc_name"]]
    .merge(
        df.groupby("sa4_code")
          .agg(chargers=("charger_id", "count"),
               dc_chargers=("charger_type", lambda s: int((s == "DC").sum())),
               plugs=("number_of_plugs", "sum"))
          .reset_index(),
        on="sa4_code", how="left",
    )
    .fillna({"chargers": 0, "dc_chargers": 0, "plugs": 0})
    .astype({"chargers": int, "dc_chargers": int, "plugs": int})
    .sort_values("chargers", ascending=False)
    .reset_index(drop=True)
)
chargers_by_sa4

,sa4_code,sa4_name,gcc_name,chargers,dc_chargers,plugs
0,118,Sydney - Eastern Suburbs,Greater Sydney,219,34,399
1,117,Sydney - City and Inner South,Greater Sydney,139,19,388
2,106,Hunter Valley exc Newcastle,Rest of NSW,127,9,361
3,101,Capital Region,Rest of NSW,117,27,411
4,103,Central West,Rest of NSW,113,17,269
5,120,Sydney - Inner West,Greater Sydney,113,17,238
6,121,Sydney - North Sydney and Hornsby,Greater Sydney,108,41,346
7,111,Newcastle and Lake Macquarie,Rest of NSW,86,13,250
8,114,Southern Highlands and Shoalhaven,Rest of NSW,76,9,203
9,112,Richmond - Tweed,Rest of NSW,76,14,213


<a id="t2-outputs"></a>
## 14. Outputs

Four datasets and the cleaning ledger are written to `data/interim/`. Task 3 reads the charger
file to augment the DC records; Task 4 reads all four to populate the DuckDB schema.

The cleaned file keeps the raw values it was derived from — `operator_raw`, `charger_rating_raw`,
`station_address_raw` — alongside the cleaned ones. That is deliberate: a cleaning decision that
cannot be checked against what it replaced cannot really be reviewed, and a reader
should be able to see both without re-running the pipeline.

In [21]:
COLUMN_ORDER = [
    "charger_id", "site_id",
    "station_name", "station_address", "lga_name", "postcode",
    "latitude", "longitude",
    "sa4_code", "sa4_name", "gcc_code", "gcc_name", "state_name", "sa4_match_method",
    "operator", "charger_type", "charger_status",
    "number_of_plugs", "power_kw_min", "power_kw_max", "connectors_in_rating",
    "rating_format", "charger_rating_raw", "operator_raw", "charger_type_raw",
    "source_feed", "source_object_id", "postcode_reported", "station_address_raw",
    "operator_truncated_flag", "postcode_conflict_flag", "postcode_valid_flag",
    "locality_placeholder_flag", "coordinate_valid_flag",
    "type_rating_conflict_flag", "plug_count_conflict_flag", "missing_name_flag",
    "probable_duplicate_flag",
]
chargers_clean = df[[column for column in COLUMN_ORDER if column in df.columns]].copy()

chargers_clean.to_csv(CLEAN_CHARGERS_PATH, index=False)
power_ratings.to_csv(RATINGS_PATH, index=False)
probable_duplicates.to_csv(DUPLICATES_PATH, index=False)
sa4_regions_nsw.to_csv(SA4_NSW_PATH, index=False)
CLEANING_LOG_PATH.write_text(
    json.dumps(
        {"raw_records": len(raw), "clean_records": len(chargers_clean),
         "steps": cleaning_log},
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

for path in (CLEAN_CHARGERS_PATH, RATINGS_PATH, DUPLICATES_PATH, SA4_NSW_PATH, CLEANING_LOG_PATH):
    print(f"  {path.relative_to(PROJECT_ROOT)}  ({path.stat().st_size / 1024:,.1f} KB)")

con.close()
pd.DataFrame(cleaning_log)

  data/interim/ev_chargers_clean.csv  (706.8 KB)
  data/interim/charger_power_ratings.csv  (20.0 KB)
  data/interim/probable_duplicates.csv  (0.7 KB)
  data/interim/sa4_regions_nsw.csv  (2.3 KB)
  data/interim/cleaning_log.json  (3.1 KB)


,step,detail,records_affected
0,whitespace normalised,cells whose text was rewritten,808
1,empty -> NA,empty strings converted to missing values,3638
2,operator canonicalised,values rewritten to a canonical name,181
3,operator truncated,unresolved truncations flagged for review,19
4,type/status separated,'Upcoming' moved into charger_status,98
5,addresses normalised,addresses reduced to one clean line,1958
6,postcode recovered,postcode recovered from the address text,120
7,postcode conflict,PCODE disagrees with the address postcode,35
8,postcode invalid,postcode outside the NSW ranges,3
9,locality placeholder,'Sydney' used in place of the real suburb,399


### 14.1 What Task 2 changed

| Issue found in Task 1 | How Task 2 handled it |
|---|---|
| Embedded newlines in 733 addresses | Collapsed to single-line form before any string comparison |
| 50 operator strings for ~35 entities | Explicit lookup table; unresolvable truncations flagged, not guessed |
| `Charger_rating` mixing 4 formats | Parsed to `power_kw_min` / `power_kw_max` plus a per-connector table |
| `Upcoming` stored as a charger type | Split into `charger_type` and `charger_status` |
| `PCODE` null in 121 rows, wrong in others | Postcode taken from the address text; conflicts flagged |
| Locality recorded as the placeholder `Sydney` | Flagged, not repaired; SA4 supplies the reliable geography |
| Duplicate coordinates | Two-pass dedup: exact removal, then rule-based reconciliation |
| The same charger in two feeds, 0.1–42 m apart | Flagged (`probable_duplicate_flag`), with the pair evidence in `probable_duplicates.csv`; not merged |
| `SA4` codes with no boundary (197, 199) | Kept, marked `has_boundary = False`; excluded from coverage |
| CRS mismatch (GDA2020 vs WGS84) | Reprojected explicitly with `always_xy` before the join |

### 14.2 What remains open, and why

Some issues are recorded rather than fixed:

* **Truncated operator names with no full form in the data.** `University of`, `Energy Austra` and
  `Fast Cities A` are flagged. The last two are probably EnergyAustralia and Fast Cities Australia,
  but nothing in the release confirms it. Their records are all Upcoming sites, which Task 3 does not
  search, so no external source confirms it either.
* **Probable duplicates across feeds.** Flagged, not merged, for the reasons in section 9.1. An
  analysis that needs one row per physical charger should exclude `probable_duplicate_flag` records.
* **The 522 `AC` placeholder ratings.** No power figure exists in the source for these. Task 3's
  augmentation from Open Charge Map or operator sites is the natural place to recover them.
* **Placeholder localities.** Only reverse geocoding could repair the suburb, and the SA4 field now
  serves the purpose the locality would have served, so it is not worth the API budget.
* **Plug counts that disagree with compound ratings.** Both figures come from the same source with
  no tiebreaker; the disagreement is flagged so a query can exclude those records rather than
  averaging away a real contradiction.

Task 3 continues from `data/interim/ev_chargers_clean.csv`, augmenting the DC records.